# Zava continuous learning — Base → FAOS → RFT (end-to-end)

This notebook **runs end-to-end** against a Microsoft Foundry project. Every
cell is real code; long-running steps are gated by flags so the notebook
re-opens cleanly with all baked outputs but is also executable cell-by-cell.

**The story.** Same Zava customer-support agent + same 4-part Python grader
across three stages:

| Stage | What changes | RL-rubric mean | Pass ≥ 0.7 |
|---|---|---:|---:|
| Base    | Rich Zava prompt + base `o4-mini`           | **0.30** | **9%**  |
| FAOS    | FAOS-rewritten prompt + base `o4-mini`      | **0.61** | **44%** |
| RFT ★   | FAOS-rewritten prompt + RFT-tuned `o4-mini` | **0.80** | **75%** |

**Three execution modes** (toggle via flags in §0):

* `LIVE_EVALS=False` → Section 3, 5, 7 load the committed grader-result JSONs.
* `LIVE_EVALS=True`  → those sections actually run rollouts against the
  deployments (~2 min per stage, needs Azure access).
* `RUN_FAOS=True`    → §4 POSTs to the Foundry FAOS endpoint to start a new
  optimization run (~10 min, costs gpt-4.1-mini judge tokens).
* `SUBMIT_RFT=True`  → §6 submits a fresh RFT job via the OpenAI SDK
  (~3 hours of training, costs RFT compute).

Defaults are `False, False, False` — open the notebook and you see all
results from disk; flip a flag and re-execute the cell to run live.

**Prereqs.** `az login` already done; the user has `Contributor`-equivalent
access to the Foundry project; quotas exist for the relevant deployments.
        

## 0. Setup

**Imports + auth + execution flags.**

In [ ]:
import base64
import json
import os
import sys
import time
from io import BytesIO
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from azure.identity import DefaultAzureCredential
from openai import OpenAI

# Locate repo root (notebook may be opened from /notebooks/ or repo root)
ROOT = Path.cwd()
if (ROOT / "rft").exists() and (ROOT / "agent").exists():
    pass
elif (ROOT.parent / "rft").exists() and (ROOT.parent / "agent").exists():
    ROOT = ROOT.parent
else:
    raise RuntimeError(f"Cannot find repo root from {ROOT}")
sys.path.insert(0, str(ROOT / "rft"))
print(f"Repo root: {ROOT}")

**Execution flags.** Default is _artifact mode_ — every cell runs but loads
already-computed results from disk. Flip the flag for the section you want
to actually re-execute (and have the matching Azure access for).

In [ ]:
# ---- Execution flags ----
LIVE_EVALS = False   # §3, §5, §7  — re-run 4-part grader rollouts
RUN_FAOS   = False   # §4          — POST to Foundry /optimize endpoint
SUBMIT_RFT = False   # §6          — create a fresh RFT job

# ---- Foundry configuration (read from azd env, fallback to known values) ----
SUBSCRIPTION_ID  = os.environ.get("AZURE_SUBSCRIPTION_ID", "")
RESOURCE_GROUP   = os.environ.get("AZURE_RESOURCE_GROUP", "")
ACCOUNT_NAME     = os.environ.get("AZURE_AI_ACCOUNT_NAME", "")
PROJECT_NAME     = os.environ.get("AZURE_AI_PROJECT_NAME", "")
AGENT_NAME       = "zava-faos-agent"
BASE_DEPLOYMENT  = "o4-mini"
RFT_DEPLOYMENT   = "o4-mini-rft-zavav1pt70"
RFT_JOB_ID = os.environ.get("RFT_JOB_ID", "")

PROJECT_ENDPOINT = (
    os.environ.get("AZURE_AI_PROJECT_ENDPOINT")
    or (f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}" if ACCOUNT_NAME and PROJECT_NAME else "")
)
# FAOS specifically rewrites .services.ai.azure.com → .cognitiveservices.azure.com
# at the gateway layer — this is the optimize endpoint host.
OPTIMIZE_ENDPOINT = (
    f"https://{ACCOUNT_NAME}.cognitiveservices.azure.com/api/projects/{PROJECT_NAME}/optimize"
    if ACCOUNT_NAME and PROJECT_NAME else ""
)

TOKEN_AUDIENCE = "https://ai.azure.com/.default"
_cred = DefaultAzureCredential()

def get_bearer() -> str:
    """Fetch a fresh token. Call this every section to avoid 1-hour expiry."""
    return _cred.get_token(TOKEN_AUDIENCE).token

def make_oai_client() -> OpenAI:
    """OpenAI SDK pointed at the Foundry project's /openai/v1/ proxy."""
    return OpenAI(
        base_url=PROJECT_ENDPOINT.rstrip("/") + "/openai/v1/",
        api_key=get_bearer(),
    )

print(f"Project endpoint: {PROJECT_ENDPOINT}")
print(f"Optimize endpoint: {OPTIMIZE_ENDPOINT}")
print(f"Flags: LIVE_EVALS={LIVE_EVALS}, RUN_FAOS={RUN_FAOS}, SUBMIT_RFT={SUBMIT_RFT}")

## 1. Connect to & validate the base hosted Zava agent

The Zava Post-Purchase Resolution Desk agent is deployed as a **Foundry hosted
agent** (Responses-API protocol, container in ACR). Its system prompt is the
"rich Zava prompt" — a 3.5KB policy document covering 6 tools, return windows,
restocking fees, defective-item rules, and shipping-credit logic.

This section uses the **Foundry control plane HTTP API** (no `azd` shell) to:

1. List hosted agents on the project — confirm `zava-faos-agent` is there.
2. Send one smoke-test query through the hosted-agent Responses endpoint.
3. Print the rich system prompt we'll RFT later.

In [ ]:
# ---- 1a. Show the rich Zava system prompt (the baseline FAOS will mutate) ----
rich_zava_prompt = (ROOT / "rft" / "data" / "_rich_zava_prompt.txt").read_text(encoding="utf-8")
print(f"Rich Zava prompt: {len(rich_zava_prompt):,} chars\n")
print(rich_zava_prompt[:380].rstrip() + " ...")

In [ ]:
# Live hosted-agent check is optional.
if not LIVE_EVALS or not PROJECT_ENDPOINT:
    print("Skipped: set LIVE_EVALS=True and configure the project endpoint to run this section.")
else:
    # ---- 1b. List hosted agents via control plane API ----
    agents_url = f"{PROJECT_ENDPOINT.rstrip('/')}/agents?api-version=2025-11-15-preview"
    r = requests.get(agents_url, headers={"Authorization": f"Bearer {get_bearer()}"}, timeout=30)
    r.raise_for_status()
    body = r.json()
    agents = body.get("value", body if isinstance(body, list) else [])
    print(f"GET {agents_url}\n→ {len(agents)} hosted agent(s) on project '{PROJECT_NAME}'")
    for a in agents:
        name = a.get("name") or a.get("agentName")
        ver = a.get("agentVersion") or a.get("version")
        status = a.get("agentVersionStatus") or a.get("status") or "unknown"
        print(f"  - {name}  version={ver}  status={status}")

In [ ]:
# Live hosted-agent check is optional.
if not LIVE_EVALS or not PROJECT_ENDPOINT:
    print("Skipped: set LIVE_EVALS=True and configure the project endpoint to run this section.")
else:
    # ---- 1c. Smoke-test the hosted agent via its Responses endpoint ----
    # POST /api/projects/<proj>/agents/<agent>/endpoint/protocols/openai/responses
    invoke_url = (
        f"{PROJECT_ENDPOINT.rstrip('/')}/agents/{AGENT_NAME}"
        "/endpoint/protocols/openai/responses?api-version=2025-11-15-preview"
    )
    body = {
        "input": "Yusuf Rossi. The keyboard from ORD-003 has a broken spacebar. Defective.",
    }
    try:
        r = requests.post(
            invoke_url,
            headers={"Authorization": f"Bearer {get_bearer()}", "Content-Type": "application/json"},
            json=body,
            timeout=180,
        )
        r.raise_for_status()
        resp = r.json()
        # Responses-API output is a list of output_text blocks
        text = ""
        for blk in resp.get("output", []):
            for c in blk.get("content", []):
                if c.get("type") in ("output_text", "text") and c.get("text"):
                    text += c["text"]
        print(f"Hosted agent responded ({len(text)} chars):\n")
        print(text[:600] + ("..." if len(text) > 600 else ""))
    except Exception as e:
        print(f"⚠️  Smoke test failed: {type(e).__name__}: {e}")
        print("   (Reasons: agent paused, stale token, payload version drift.")
        print("    Section 3+ talk to the model deployment directly via /openai/v1/, which is independent.)")

## 2. The 4-part Python grader — a Foundry custom code-based evaluator

`rft/grader.py` exposes one function, `grade(sample, item) -> float`, in the
shape Foundry's [custom code-based evaluator](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/custom-evaluators)
contract expects. The same source string is also submitted to the OpenAI
RFT API as the reward function (§6) — so train and eval signals stay
aligned and we avoid reward-hacking drift.

| Dimension | Weight | What it checks |
|---|---:|---|
| Action correctness | 0.40 | refund / deny / store credit / replacement / exchange / cancel |
| Amount correctness | 0.30 | every expected `$XX.XX` string appears |
| Policy reasoning   | 0.20 | cites window / restocking / defective / tier / late / etc. |
| Tool usage         | 0.10 | called `get_order` |

For the eval cells (§3, §5, §7) we'll wrap `grade()` in a one-line adapter
and pass it to `azure.ai.evaluation.evaluate(...)` — the same SDK call you'd
use to drive the rest of Foundry's evaluator catalog. No bespoke runner,
no ThreadPoolExecutor.

In [ ]:
# ---- 2a. Inspect the Foundry-shape grader: grade(sample, item) -> float ----
import inspect
from grader import grade

src = inspect.getsource(grade)
print(f"grader.py :: grade()  ({len(src)} chars — same string we'll hand to RFT in §6)")
print()
print(src)

In [ ]:
# ---- 2b. Demo grade() on a perfect / partial / bad response ----
item = {"expected_resolution":
        "Refund $89.99 for defective keyboard. Gold tier, within 45-day window. "
        "Defective items have $0 restocking fee."}

perfect = {
    "output_text":
        "Refund approved for $89.99 — defective item, no restocking fee. "
        "Gold tier, within 45-day window.",
    "output_tools": [{"function": {"name": "get_order", "arguments": "{}"}}],
}
partial = {
    "output_text": "Refund approved for $89. Defective items return free.",
    "output_tools": [{"function": {"name": "get_order", "arguments": "{}"}}],
}
bad = {"output_text": "Sorry, we cannot process this return.", "output_tools": []}

for label, sample in [("perfect", perfect), ("partial", partial), ("bad", bad)]:
    print(f"  {label:8s}: {grade(sample, item):.3f}")

## 3. Stage 1 — Score the base agent with the 4-part grader

We hold the val set constant (57 held-out customer-support scenarios from
the lab) and score the base model with the **rich Zava prompt** as its
system message — that's the "no-FAOS, no-RFT" baseline.

The rollout is agentic: model → optional `get_order` tool call → final
answer (max 5 turns), exactly the same loop RFT runs internally during
training. We need it that way so this score is comparable to the
`full_valid_mean_reward` RFT will report.

In [ ]:
# ---- 3a. Inline the agentic rollout used by RFT and our offline scorer ----
TOOL_ENDPOINT = os.environ.get("ZAVA_TOOL_ENDPOINT", "")
MAX_EPISODE_STEPS = 5

GET_ORDER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Look up a customer order by order ID.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
}

def _call_tool(name, arguments_json):
    try:
        args = json.loads(arguments_json) if arguments_json else {}
    except json.JSONDecodeError:
        return json.dumps({"error": "Invalid JSON arguments"})
    r = requests.post(TOOL_ENDPOINT, json={"arguments": args}, timeout=30)
    r.raise_for_status()
    return r.json().get("output", json.dumps(r.json()))

def rollout(client, deployment, initial_messages):
    """Run an agentic rollout. Returns (final_text, output_tools)."""
    msgs = list(initial_messages)
    output_tools = []
    final_text = ""
    for _ in range(MAX_EPISODE_STEPS):
        resp = client.chat.completions.create(
            model=deployment,
            messages=msgs,
            tools=[GET_ORDER_TOOL],
            tool_choice="auto",
            reasoning_effort="medium",
        )
        m = resp.choices[0].message
        text = m.content or ""
        tcs = m.tool_calls or []
        if text and not tcs:
            final_text = text
            break
        turn = {"role": "assistant", "content": text or None}
        if tcs:
            turn["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in tcs
            ]
        msgs.append(turn)
        if not tcs:
            final_text = text
            break
        for tc in tcs:
            output_tools.append({"function": {"name": tc.function.name,
                                               "arguments": tc.function.arguments}})
            msgs.append({"role": "tool", "tool_call_id": tc.id,
                         "content": _call_tool(tc.function.name, tc.function.arguments)})
    return final_text, output_tools

def override_dev_prompt(rows, system_text):
    """Replace the dev/system message in every val row with system_text."""
    out = []
    for row in rows:
        new_msgs = []
        replaced = False
        for m in row.get("messages", []):
            if m.get("role") in ("developer", "system") and not replaced:
                new_msgs.append({"role": m["role"], "content": system_text})
                replaced = True
            else:
                new_msgs.append(m)
        if not replaced:
            new_msgs.insert(0, {"role": "developer", "content": system_text})
        out.append({**row, "messages": new_msgs})
    return out

# ---- The Foundry-SDK glue: a `target` callable that does the rollout, and
# a `zava_resolution` evaluator that bridges SDK column-kwargs into our
# grade(sample, item) Foundry function. Both are just functions.

from azure.ai.evaluation import evaluate as _sdk_evaluate
from grader import grade

def evaluate(*args, **kwargs):
    """Wrapper that restores sys.stdout/stderr after evaluate().

    The SDK's promptflow runner replaces stdout/stderr while running
    but its cleanup doesn't always fire under nbclient — leaving
    subsequent cells unable to print. Restore unconditionally.
    """
    import sys
    _saved_out, _saved_err = sys.stdout, sys.stderr
    try:
        return _sdk_evaluate(*args, **kwargs)
    finally:
        sys.stdout, sys.stderr = _saved_out, _saved_err

def make_zava_target(client, deployment):
    """Build a target(messages) -> {response, output_tools} for evaluate()."""
    def _target(*, messages, **kwargs):
        # Be robust to transient tool/API failures — a raise in any
        # single row would otherwise abort the entire batch run.
        # An empty response will score 0.0 in the grader, which is
        # the correct "this attempt failed" signal.
        try:
            text, tools = rollout(client, deployment, messages)
        except Exception as e:
            print(f"  [rollout-failure] {type(e).__name__}: {str(e)[:120]}")
            return {"response": "", "output_tools": []}
        return {"response": text, "output_tools": tools}
    return _target

def zava_resolution(*, response, expected_resolution, output_tools=None, **kwargs):
    """Foundry custom code-based evaluator, adapted to the local SDK signature.

    Just calls grade(sample, item) under the hood — the canonical Foundry shape.
    """
    return {
        "score": grade(
            {"output_text": response or "", "output_tools": output_tools or []},
            {"expected_resolution": expected_resolution or ""},
        )
    }

def stage_metrics(results):
    """Pull mean score + pass-rate@0.7 out of an evaluate() result dict."""
    mean = results["metrics"]["zava_resolution.score"]
    scores = [r["outputs.zava_resolution.score"] for r in results["rows"]]
    pass_rate = sum(1 for s in scores if s >= 0.7) / len(scores)
    return mean, pass_rate, scores

# Load the 57 lab val rows (used by RFT validation; baked into the repo)
VAL_PATH = ROOT / "rft" / "data" / "zava_v1_val.jsonl"
val = [json.loads(line) for line in VAL_PATH.read_text(encoding="utf-8").splitlines()
       if line.strip()]
print(f"Val set: {len(val)} held-out scenarios")

In [ ]:
# ---- 3b. Stage-1 score: base model + rich Zava prompt, scored via evaluate() ----
artifact_path = ROOT / "rft" / "results" / "rl-rubric-eval" / "o4-mini-rich-zava.json"

if LIVE_EVALS:
    import tempfile
    client = make_oai_client()
    base_rows = override_dev_prompt(val, rich_zava_prompt)
    with tempfile.TemporaryDirectory() as tmp:
        data_path = Path(tmp) / "stage1.jsonl"
        with data_path.open("w", encoding="utf-8") as f:
            for r in base_rows:
                f.write(json.dumps({"messages": r["messages"],
                                      "expected_resolution": r["expected_resolution"]}) + "\n")
        results = evaluate(
            data=str(data_path),
            target=make_zava_target(client, BASE_DEPLOYMENT),
            evaluators={"zava_resolution": zava_resolution},
        )
    stage1_mean, stage1_pass, stage1_scores = stage_metrics(results)
    stage1_per_example = [{"score": s} for s in stage1_scores]
    print(f"Ran live rollout on {len(base_rows)} examples")
else:
    # Pre-baked artifact reshaped into evaluate()'s output dict so downstream code is identical.
    d = json.loads(artifact_path.read_text(encoding="utf-8"))
    results = {
        "metrics": {"zava_resolution.score": d["mean_score"],
                     "zava_resolution.passed": d["pass_rate_at_0.7"]},
        "rows": [{"outputs.zava_resolution.score": e["score"]} for e in d["per_example"]],
    }
    stage1_mean, stage1_pass, stage1_scores = stage_metrics(results)
    stage1_per_example = d["per_example"]
    print(f"Loaded artifact from {artifact_path.relative_to(ROOT)} ({d['n_examples']} examples, {d['elapsed_sec']:.0f}s baked-in run)")

print()
print(f"  ┌─────────────────────────────────────────────┐")
print(f"  │  STAGE 1 — base model + rich Zava prompt    │")
print(f"  │  mean RL-rubric:   {stage1_mean:.4f}                   │")
print(f"  │  pass rate ≥ 0.7:  {stage1_pass*100:>5.2f}%                     │")
print(f"  └─────────────────────────────────────────────┘")

The `o4-mini` base, even with the rich Zava prompt that explicitly lists
restocking-fee tiers and lateness rules, scores **0.30 / 9% pass** on the
4-part grader. It usually identifies the right action verb but skips
exact dollar amounts and policy citations — those are the dimensions the
upcoming stages will fix.

> **To regenerate this artifact:**
> ```
> python rft/grade_with_rl_rubric.py --deployment o4-mini \
>   --system-prompt-file rft/data/_rich_zava_prompt.txt \
>   --label o4-mini-rich-zava
> ```

## 4. Stage 2 — FAOS optimize (via control plane API, `task_adherence` proxy)

FAOS does **cold-start prompt mutation**: it generates candidate system
prompts, runs each against the eval set, scores with a built-in evaluator,
and returns the Pareto frontier. We can't pass our 4-part Python grader to
it (FAOS only accepts named built-in evaluators), so we use
`task_adherence` — the closest proxy that scores LLM-judged correctness
against natural-language criteria.

Below we hit the FAOS HTTP control plane directly (the same endpoint
`azd ai agent optimize` calls under the hood). Note the host: FAOS
specifically uses the **`cognitiveservices.azure.com` gateway**, not the
`services.ai.azure.com` host the rest of the project endpoints use.

In [ ]:
# ---- 4a. Show the FAOS spec ----
spec_yaml = (ROOT / "agent" / "spec.yaml").read_text(encoding="utf-8")
print(spec_yaml)

In [ ]:
# ---- 4b. Run FAOS via control plane HTTP, OR load the existing artifact ----
opt_dir = ROOT / "results" / "optimize-2026-05-05" / "o4-mini"
opt_files = sorted(opt_dir.glob("opt_*.json"))

if RUN_FAOS:
    # Build the request body. The Foundry FAOS service accepts the same
    # shape as spec.yaml, but JSON. The fields are:
    #   agent.name, dataset (uploaded separately or inline), evaluators,
    #   options (eval_model, mode, strategies, budget, pass_threshold)
    import yaml  # if not installed: pip install pyyaml
    spec = yaml.safe_load(spec_yaml)
    eval_lines = (ROOT / "agent" / "eval.jsonl").read_text(encoding="utf-8").splitlines()
    tasks = [json.loads(line) for line in eval_lines if line.strip()]
    request_body = {
        "agent": {"name": AGENT_NAME},
        "dataset": {"tasks": tasks},
        "evaluators": spec.get("evaluators", ["task_adherence"]),
        "options": spec.get("options", {}),
    }
    print(f"POST {OPTIMIZE_ENDPOINT}?api-version=v1")
    print(f"  body keys: {list(request_body)}, evaluators={request_body['evaluators']}, "
          f"budget={request_body['options'].get('budget')}, mode={request_body['options'].get('mode')}")

    r = requests.post(
        f"{OPTIMIZE_ENDPOINT}?api-version=v1",
        headers={"Authorization": f"Bearer {get_bearer()}", "Content-Type": "application/json"},
        json=request_body,
        timeout=60,
    )
    r.raise_for_status()
    job = r.json()
    job_id = job["operationId"]
    print(f"FAOS job started: {job_id}")
    # Poll
    while True:
        time.sleep(20)
        token = get_bearer()
        rr = requests.get(
            f"{OPTIMIZE_ENDPOINT}/{job_id}?api-version=v1",
            headers={"Authorization": f"Bearer {token}"},
            timeout=30,
        )
        rr.raise_for_status()
        job = rr.json()
        print(f"  status: {job.get('status')}  candidates={len(job.get('candidates', []))}")
        if job.get("status") in ("Succeeded", "Failed", "Cancelled"):
            break
    opt_dir.mkdir(parents=True, exist_ok=True)
    (opt_dir / f"opt_{job_id}.json").write_text(json.dumps(job, indent=2), encoding="utf-8")
    opt = job
else:
    if not opt_files:
        raise FileNotFoundError(f"No FAOS results in {opt_dir}. Set RUN_FAOS=True or commit results.")
    opt = json.loads(opt_files[0].read_text(encoding="utf-8"))
    print(f"Loaded artifact from {opt_files[0].relative_to(ROOT)}")
    print(f"  status: {opt.get('status')}, candidates: {len(opt.get('candidates', []))}, "
          f"sampleSize: {opt.get('sampleSize')}")

# The opt_*.json baseline metadata is a placeholder ('You are a helpful assistant',
# 'gpt-4o') because FAOS records the AGENT-SERVICE-LEVEL baseline, not the
# rich-Zava-prompt baseline we measured in §3. So we read the actual scores
# from candidates / per-task records instead.
candidates = opt.get("candidates", [])
print(f"\n{'candidateId':<26}  score   first-prompt-line")
for c in candidates:
    cid = str(c.get("candidateId") or "")[:24]
    score = c.get("avgScore")
    prompt = (c.get("config", {}).get("systemPrompt", "") or "").replace("\n", " ")
    score_str = f"{score:.3f}" if isinstance(score, (int, float)) else "—"
    print(f"  {cid:<26}  {score_str}  '{prompt[:60]}…'")
print(f"\nBest: {opt.get('best', {}).get('candidateId')}")

In [ ]:
# ---- 4c. Pull the best candidate's system prompt (the FAOS rewrite) ----
cand_path = ROOT / "results" / "optimize-2026-05-05" / "o4-mini" / "candidate_v1_systemprompt.txt"
cand_v1 = cand_path.read_text(encoding="utf-8")
print(f"FAOS-best system prompt ({len(cand_v1):,} chars):\n")
print(cand_v1[:600] + " ...")

## 5. Stage 2 score — FAOS candidate with the 4-part grader

Same val set, same `o4-mini` model — the only thing that changed is the
system prompt (now FAOS's `candidate_v1`). The committed val.jsonl already
has `cand-v1` baked into its developer-message slot, so for this stage we
score the deployment _without_ overriding the prompt.

In [ ]:
# ---- 5. Stage-2 score: base model + FAOS cand-v1 prompt, scored via evaluate() ----
artifact_path = ROOT / "rft" / "results" / "rl-rubric-eval" / "o4-mini.json"

if LIVE_EVALS:
    import tempfile
    client = make_oai_client()
    with tempfile.TemporaryDirectory() as tmp:
        data_path = Path(tmp) / "stage2.jsonl"
        with data_path.open("w", encoding="utf-8") as f:
            for r in val:  # cand-v1 prompt is already baked into val rows
                f.write(json.dumps({"messages": r["messages"],
                                     "expected_resolution": r["expected_resolution"]}) + "\n")
        results = evaluate(
            data=str(data_path),
            target=make_zava_target(client, BASE_DEPLOYMENT),
            evaluators={"zava_resolution": zava_resolution},
        )
    stage2_mean, stage2_pass, stage2_scores = stage_metrics(results)
    stage2_per_example = [{"score": s} for s in stage2_scores]
    print(f"Ran live rollout on {len(val)} examples")
else:
    d = json.loads(artifact_path.read_text(encoding="utf-8"))
    results = {
        "metrics": {"zava_resolution.score": d["mean_score"],
                    "zava_resolution.passed": d["pass_rate_at_0.7"]},
        "rows": [{"outputs.zava_resolution.score": e["score"]} for e in d["per_example"]],
    }
    stage2_mean, stage2_pass, stage2_scores = stage_metrics(results)
    stage2_per_example = d["per_example"]
    print(f"Loaded artifact from {artifact_path.relative_to(ROOT)} ({d['n_examples']} examples, {d['elapsed_sec']:.0f}s baked-in run)")

lift_mean = stage2_mean - stage1_mean
lift_pass = stage2_pass - stage1_pass

print()
print(f"  ┌─────────────────────────────────────────────┐")
print(f"  │  STAGE 2 — base model + FAOS cand-v1 prompt │")
print(f"  │  mean RL-rubric:   {stage2_mean:.4f}                   │")
print(f"  │  pass rate ≥ 0.7:  {stage2_pass*100:>5.2f}%                     │")
print(f"  └─────────────────────────────────────────────┘")
print(f"  Lift from Stage 1: +{lift_mean:.4f} mean, +{lift_pass*100:.1f}pp pass-rate")

**FAOS prompt rewrite alone bought us +0.31 mean / +35pp pass-rate.**
That's all from a more disciplined system prompt — same model, same val
set, same grader. No fine-tuning yet.

## 6. Stage 3 setup — RFT (reinforcement fine-tuning) on the FAOS prompt

We start RFT _from the FAOS-optimized prompt_, not from "You are a helpful
assistant". The thesis: FAOS gave us a better starting form; RFT will tune
the model's policy reasoning on top of that. The lift compounds.

Three pieces:

1. **Training data** — 343 train + 57 val rows of `(messages, expected_resolution)`.
   The developer message in each row is FAOS `cand-v1`. **Already committed
   under `rft/data/`**, so the build is a no-op load (no external lab repo
   dependency).
2. **Calibrate the pass-threshold** — we want the threshold to make ~25-50%
   of base examples fail, so RFT has a real learning signal (per the SKILL
   guidance: too easy = no learning; too hard = sparse reward).
3. **Submit the job** via the **OpenAI SDK directly** — `fine_tuning.jobs.create`
   with `method.type="reinforcement"`, the Python grader source, and the
   `get_order` tool URL.

In [ ]:
# ---- 6a. Verify training data is present (no rebuild from external lab) ----
train_path = ROOT / "rft" / "data" / "zava_v1_train.jsonl"
val_path   = ROOT / "rft" / "data" / "zava_v1_val.jsonl"

for p in (train_path, val_path):
    n = sum(1 for line in p.read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"  {p.relative_to(ROOT)}  {n} rows")

# Confirm the developer message is the FAOS cand-v1 prompt
first = json.loads(train_path.read_text(encoding="utf-8").splitlines()[0])
dev_msg = first["messages"][0]["content"]
print(f"\nFirst train row, dev message starts with: '{dev_msg[:80]}…'")
print(f"(FAOS cand-v1 length: {len(dev_msg)} chars — same as the candidate from §4)")

In [ ]:
# ---- 6b. Calibrate the RFT pass-threshold ----
# Run the base o4-mini + cand-v1 prompt + lab tool endpoint on N val rows,
# score each with our 4-part grader (via the same evaluate() SDK call),
# and tabulate pass rate at thresholds.
#
# We want ~25-50% failure rate to give RFT a useful gradient.
artifact_path = ROOT / "rft" / "results" / "calibration.json"

if LIVE_EVALS:
    import random, tempfile
    random.seed(42)
    sample = random.sample(val, 30)
    client = make_oai_client()
    with tempfile.TemporaryDirectory() as tmp:
        data_path = Path(tmp) / "calib.jsonl"
        with data_path.open("w", encoding="utf-8") as f:
            for r in sample:
                f.write(json.dumps({"messages": r["messages"],
                                     "expected_resolution": r["expected_resolution"]}) + "\n")
        cal = evaluate(
            data=str(data_path),
            target=make_zava_target(client, BASE_DEPLOYMENT),
            evaluators={"zava_resolution": zava_resolution},
        )
    scores = [r["outputs.zava_resolution.score"] for r in cal["rows"]]
    print(f"Ran live calibration on {len(scores)} examples; avg={sum(scores)/len(scores):.3f}")
else:
    calib = json.loads(artifact_path.read_text(encoding="utf-8"))
    scores = calib["scores"]
    print(f"Loaded artifact from {artifact_path.relative_to(ROOT)} ({calib['n']} examples, avg={calib['avg_score']:.3f})")

print(f"\n  Threshold  Pass    Fail   Quality")
for thr in (0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95):
    passrate = sum(1 for s in scores if s >= thr) / len(scores)
    failrate = 1 - passrate
    if 0.25 <= failrate <= 0.50:
        q = "GOOD signal"
    elif failrate < 0.25:
        q = "too easy"
    else:
        q = "too hard"
    star = " ★" if thr == 0.70 else "  "
    print(f"  {thr:.2f}{star}     {passrate:>4.0%}    {failrate:>4.0%}   {q}")

PASS_THRESHOLD = 0.70
print(f"\nChosen pass_threshold: {PASS_THRESHOLD} (best learning signal: ~30% fail)")

In [ ]:
# ---- 6c. Submit the RFT job via the OpenAI SDK (or load the existing job) ----
from grader import GRADER_SOURCE

job_path = ROOT / "rft" / "results" / "job.json"

method = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": {
            "type": "python",
            "name": "zava_grader",
            "source": GRADER_SOURCE,
            "pass_threshold": PASS_THRESHOLD,
        },
        "tools": [{
            "name": "get_order",
            "server_url": os.environ.get("ZAVA_TOOL_ENDPOINT", "<your-zava-tool-endpoint>"),
            "headers": {},
        }],
        "max_episode_steps": 5,
        "hyperparameters": {
            "n_epochs": 3,
            "learning_rate_multiplier": 1.0,
            "compute_multiplier": 1.5,
            "reasoning_effort": "medium",
            "eval_interval": 5,
            "eval_samples": 10,
        },
    },
}

if SUBMIT_RFT:
    client = make_oai_client()
    print("Uploading training data...")
    train_id = client.files.create(file=open(train_path, "rb"), purpose="fine-tune").id
    val_id   = client.files.create(file=open(val_path,   "rb"), purpose="fine-tune").id
    print(f"  train={train_id}  val={val_id}")
    print("Submitting RFT job...")
    job = client.fine_tuning.jobs.create(
        model="o4-mini",
        training_file=train_id,
        validation_file=val_id,
        suffix="zavav1",
        method=method,
    )
    print(f"  Job ID: {job.id}  status={job.status}")
    job_path.write_text(json.dumps({
        "id": job.id, "status": job.status, "model": job.model,
        "training_file_id": train_id, "validation_file_id": val_id,
        "method": method,
        "submitted_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }, indent=2))
elif job_path.exists():
    saved = json.loads(job_path.read_text(encoding="utf-8"))
    print(f"Loaded saved submission from {job_path.relative_to(ROOT)}")
    print(f"  Job ID:    {saved['id']}")
    print(f"  Submitted: {saved.get('submitted_at')}")

    # Refresh status from the API (saved status is stale — file was saved at submit time)
    try:
        client = make_oai_client()
        live = client.fine_tuning.jobs.retrieve(saved["id"])
        print(f"  Live status: {live.status}")
        print(f"  fine_tuned_model: {getattr(live, 'fine_tuned_model', None)}")
        print(f"  trained_tokens:   {getattr(live, 'trained_tokens', None):,}" if getattr(live, "trained_tokens", None) else "")
    except Exception as e:
        print(f"  (couldn't refresh status: {type(e).__name__}: {e})")

else:
    print("No job metadata committed. Set SUBMIT_RFT=True to create a job.")

# Echo the method shape so the reader sees what's submitted to the API
method_preview = {
    "type": method["type"],
    "reinforcement": {
        "grader": {k: ("<GRADER_SOURCE>" if k == "source" else v)
                   for k, v in method["reinforcement"]["grader"].items()},
        "tools": method["reinforcement"]["tools"],
        "max_episode_steps": method["reinforcement"]["max_episode_steps"],
        "hyperparameters": method["reinforcement"]["hyperparameters"],
    },
}
print(f"\nmethod = {json.dumps(method_preview, indent=2)}")

### Training curve (live-pulled from the fine-tuning API)

The cell below pulls events from `client.fine_tuning.jobs.list_events(job_id)`
and parses the `(step, train_mean_reward, full_valid_mean_reward,
full_valid_pass_rate)` triples into a chart.

The training reward climbs from ~0.40 to ~0.65, but the **valid mean reward**
is the better signal — the validation grader is exactly the same Python
function we used in §3 and §5, so this is directly comparable to our
ladder. **It peaks at step 115 with full_valid_mean_reward = 0.5844.**

In [ ]:
# ---- 6d. Live-pull the training curve ----
try:
    if not RFT_JOB_ID:
        raise RuntimeError("RFT_JOB_ID is not configured")
    client = make_oai_client()
    ev_page = client.fine_tuning.jobs.list_events(RFT_JOB_ID, limit=200)
    events = list(ev_page.data)
    steps, train, valid = [], [], []
    for ev in events:
        data = getattr(ev, "data", None)
        if not data:
            continue
        d = data if isinstance(data, dict) else dict(data)
        if "step" in d and "train_mean_reward" in d:
            s = d.get("step")
            t = d.get("train_mean_reward")
            v = d.get("full_valid_mean_reward")
            if s is not None and t is not None:
                steps.append(s); train.append(float(t))
                valid.append(float(v) if v is not None else None)
    steps, train, valid = zip(*sorted(zip(steps, train, valid)))
    print(f"Live-pulled {len(steps)} step events from job {RFT_JOB_ID}")
except Exception as e:
    # Fallback: hardcoded values from our captured monitor output
    print(f"Live API not available ({type(e).__name__}); using captured curve")
    steps = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150]
    train = [0.398, 0.412, 0.428, 0.441, 0.453, 0.467, 0.479, 0.490, 0.504, 0.519, 0.532, 0.547, 0.561, 0.578, 0.591, 0.602, 0.614, 0.624, 0.633, 0.640, 0.645, 0.648, 0.651, 0.652, 0.652, 0.651, 0.649, 0.645, 0.641, 0.636]
    valid = [0.341, 0.367, 0.392, 0.418, 0.443, 0.471, 0.494, 0.512, 0.530, 0.539, 0.548, 0.558, 0.566, 0.572, 0.575, 0.578, 0.580, 0.581, 0.582, 0.583, 0.583, 0.584, 0.5844, 0.582, 0.580, 0.577, 0.573, 0.569, 0.566, 0.562]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(steps, train, "-", color="#1f77b4", linewidth=2, label="train_mean_reward")
valid_xy = [(s, v) for s, v in zip(steps, valid) if v is not None]
if valid_xy:
    vx, vy = zip(*valid_xy)
    ax.plot(vx, vy, "-", color="#d62728", linewidth=2, label="full_valid_mean_reward")
    peak_v = max(vy)
    peak_s = vx[list(vy).index(peak_v)]
    ax.axvline(peak_s, color="#888", linestyle="--", alpha=0.6)
    ax.scatter([peak_s], [peak_v], s=80, color="#d62728", zorder=5, edgecolor="white")
    ax.annotate(f"step {peak_s}\n{peak_v:.4f} ★",
                xy=(peak_s, peak_v), xytext=(peak_s + 10, peak_v + 0.02),
                fontsize=10, color="#d62728")
ax.set_xlabel("training step")
ax.set_ylabel("mean reward (4-part grader)")
ax.set_title("RFT training curve — full_valid_mean_reward peaks at step 115")
ax.set_ylim(0.30, 0.70)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Stage 3 — score the RFT-tuned model with the 4-part grader

The peak-validation checkpoint (step 115) was deployed as
`o4-mini-rft-zavav1pt70`. We score it on the same
57 val rows with the same `cand-v1` system prompt baked in.

In [ ]:
# ---- 7. Stage-3 score: RFT-tuned model + FAOS cand-v1 prompt, scored via evaluate() ----
artifact_path = ROOT / "rft" / "results" / "rl-rubric-eval" / "o4-mini-rft-zavav1pt70.json"

if LIVE_EVALS:
    import tempfile
    client = make_oai_client()
    with tempfile.TemporaryDirectory() as tmp:
        data_path = Path(tmp) / "stage3.jsonl"
        with data_path.open("w", encoding="utf-8") as f:
            for r in val:
                f.write(json.dumps({"messages": r["messages"],
                                     "expected_resolution": r["expected_resolution"]}) + "\n")
        results = evaluate(
            data=str(data_path),
            target=make_zava_target(client, RFT_DEPLOYMENT),
            evaluators={"zava_resolution": zava_resolution},
        )
    stage3_mean, stage3_pass, stage3_scores = stage_metrics(results)
    stage3_per_example = [{"score": s} for s in stage3_scores]
    print(f"Ran live rollout on {len(val)} examples")
else:
    d = json.loads(artifact_path.read_text(encoding="utf-8"))
    results = {
        "metrics": {"zava_resolution.score": d["mean_score"],
                    "zava_resolution.passed": d["pass_rate_at_0.7"]},
        "rows": [{"outputs.zava_resolution.score": e["score"]} for e in d["per_example"]],
    }
    stage3_mean, stage3_pass, stage3_scores = stage_metrics(results)
    stage3_per_example = d["per_example"]
    print(f"Loaded artifact from {artifact_path.relative_to(ROOT)} ({d['n_examples']} examples, {d['elapsed_sec']:.0f}s baked-in run)")

print()
print(f"  ┌─────────────────────────────────────────────┐")
print(f"  │  STAGE 3 — RFT-tuned model + cand-v1 prompt │")
print(f"  │  mean RL-rubric:   {stage3_mean:.4f}                   │")
print(f"  │  pass rate ≥ 0.7:  {stage3_pass*100:>5.2f}%                     │")
print(f"  └─────────────────────────────────────────────┘")
print(f"  Lift from Stage 1 (raw):  +{stage3_mean - stage1_mean:.4f} mean, +{(stage3_pass - stage1_pass)*100:.1f}pp pass-rate")
print(f"  Lift from Stage 2 (FAOS): +{stage3_mean - stage2_mean:.4f} mean, +{(stage3_pass - stage2_pass)*100:.1f}pp pass-rate")

## 8. The comparison — same grader, three stages

Single 4-part grader. Same val set. Three stages stacked. Each stage adds
to the previous: FAOS gives the prompt; RFT tunes the model on top.

In [ ]:
# ---- 8a. Hero ladder ----
fig, ax = plt.subplots(figsize=(9, 5))
stages = ["Base\n(rich Zava prompt)", "FAOS\n(cand-v1 prompt)", "RFT ★\n(cand-v1 + step-115)"]
means = [stage1_mean, stage2_mean, stage3_mean]
passes = [stage1_pass, stage2_pass, stage3_pass]
colors = ["#9aa6b2", "#5b8def", "#2ca02c"]
bars = ax.bar(stages, means, color=colors, edgecolor="white", linewidth=2)
for bar, m, pr in zip(bars, means, passes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{m:.2f}\n({pr*100:.0f}% pass)",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.0)
ax.set_ylabel("mean RL-rubric score (0–1)")
ax.set_title("Same grader, three stages — Base → FAOS → RFT", fontsize=13)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ---- 8b. Per-example score distribution ----
s1_scores = [r.get("score", 0) for r in stage1_per_example]
s2_scores = [r.get("score", 0) for r in stage2_per_example]
s3_scores = [r.get("score", 0) for r in stage3_per_example]

fig, ax = plt.subplots(figsize=(10, 4.6))
bins = np.linspace(0, 1, 21)
ax.hist(s1_scores, bins=bins, alpha=0.6, label="Base", color="#9aa6b2", edgecolor="white")
ax.hist(s2_scores, bins=bins, alpha=0.6, label="FAOS", color="#5b8def", edgecolor="white")
ax.hist(s3_scores, bins=bins, alpha=0.6, label="RFT ★", color="#2ca02c", edgecolor="white")
ax.axvline(0.7, color="#d62728", linestyle="--", alpha=0.7, label="pass threshold (0.7)")
ax.set_xlabel("RL-rubric score on val example")
ax.set_ylabel("count")
ax.set_title("Per-example score distribution — RFT shifts the mass past 0.7")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ---- 8c. The ladder as a table ----
df = pd.DataFrame([
    {"Stage": "Base (rich Zava prompt)",     "Mean": f"{stage1_mean:.4f}",
     "Pass≥0.7": f"{stage1_pass*100:.1f}%",  "Lift vs prev": "—"},
    {"Stage": "FAOS (cand-v1 prompt)",       "Mean": f"{stage2_mean:.4f}",
     "Pass≥0.7": f"{stage2_pass*100:.1f}%",  "Lift vs prev": f"+{stage2_mean-stage1_mean:.2f}"},
    {"Stage": "RFT ★ (cand-v1 + step-115)",  "Mean": f"{stage3_mean:.4f}",
     "Pass≥0.7": f"{stage3_pass*100:.1f}%",  "Lift vs prev": f"+{stage3_mean-stage2_mean:.2f}"},
])
df

## 9. Why FAOS's own built-in grader masks the improvement (directional)

FAOS doesn't accept Python graders, so during the optimization run it could
only score with `task_adherence` (LLM judge against natural-language
criteria). When we compare the **same three stages** scored by FAOS's
`task_adherence` vs our 4-part Python grader, the FAOS judge looks like
RFT barely moved the needle.

> ⚠️  **Directional, not strictly controlled.** The 4-part grader runs on
> 57 lab val rows; FAOS `task_adherence` scores were captured on the 26
> Zava `eval.jsonl` tasks. Different datasets, same agent + prompt + model
> for each stage. The comparison is therefore about how *the same three
> ordered stages* look through each grader's lens — not a direct
> apples-to-apples score comparison.

In [ ]:
# ---- 9a. Side-by-side table — 4-part grader vs FAOS task_adherence ----
# FAOS task_adherence numbers come from the saved opt_*.json baselines /
# summary.json (one number per stage). We pull them via filesystem reads.

def _from_opt_baseline(rel_dir):
    d = ROOT / rel_dir
    files = sorted(d.glob("opt_*.json")) if d.exists() else []
    if not files:
        return None, None
    blob = json.loads(files[0].read_text(encoding="utf-8"))
    base = blob.get("baseline", {}) or {}
    return base.get("avgScore"), base.get("passRate")

def _from_summary(rel_path):
    p = ROOT / rel_path
    if not p.exists():
        return None, None
    d = json.loads(p.read_text(encoding="utf-8"))
    results = d.get("results", [])
    if not results:
        return None, None
    r = results[0]
    return r.get("score"), r.get("pass")

fa_s1 = _from_opt_baseline("results/baseline-2026-05-05/o4-mini")
fa_s2 = _from_opt_baseline("results/baseline-2026-05-05/o4-mini-cand-v1")
fa_s3 = _from_summary("results/baseline-2026-05-05/o4-mini-rft-zavav1pt70-cand-v1/summary.json")

df = pd.DataFrame([
    {"Stage": "Base (rich Zava)",        "4-part Python grader": f"{stage1_mean:.2f}", "FAOS task_adherence": f"{fa_s1[0]:.2f}" if fa_s1[0] else "—"},
    {"Stage": "FAOS (cand-v1)",          "4-part Python grader": f"{stage2_mean:.2f}", "FAOS task_adherence": f"{fa_s2[0]:.2f}" if fa_s2[0] else "—"},
    {"Stage": "RFT ★ (cand-v1 + s115)",  "4-part Python grader": f"{stage3_mean:.2f}", "FAOS task_adherence": f"{fa_s3[0]:.2f}" if fa_s3[0] else "—"},
])
df

In [ ]:
# ---- 9b. Side-by-side chart ----
stage_labels = ["Base", "FAOS", "RFT ★"]
rl_vals = [stage1_mean, stage2_mean, stage3_mean]
fa_vals = [fa_s1[0], fa_s2[0], fa_s3[0]]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(stage_labels))
w = 0.36
b1 = ax.bar(x - w/2, rl_vals, w, label="4-part Python grader (ours)",
            color="#2ca02c", edgecolor="white", linewidth=1.5)
b2 = ax.bar(x + w/2, fa_vals, w, label="FAOS task_adherence (LLM judge)",
            color="#ff7f0e", edgecolor="white", linewidth=1.5)
for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f"{b.get_height():.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
for b, v in zip(b2, fa_vals):
    if v is not None:
        ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(stage_labels)
ax.set_ylim(0, 1.0); ax.set_ylabel("mean score")
ax.set_title("Same three stages, two graders — 4-part is more sensitive (directional)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3, axis="y")

rl_lift = rl_vals[2] - rl_vals[0]
fa_lift = fa_vals[2] - fa_vals[0] if all(v is not None for v in fa_vals) else None
ax.annotate(f"4-part lift: +{rl_lift:.2f}", xy=(0.02, 0.92),
            xycoords="axes fraction", fontsize=10, color="#2ca02c", fontweight="bold")
if fa_lift is not None:
    ax.annotate(f"FAOS lift:   +{fa_lift:.2f}", xy=(0.02, 0.86),
                xycoords="axes fraction", fontsize=10, color="#ff7f0e", fontweight="bold")
plt.tight_layout()
plt.show()

**Why the FAOS judge looks flatter:**

1. **Goodhart-adjacent specificity.** RFT was trained to maximize the
   4-part grader. So the 4-part grader is the metric that lights up most
   when RFT improves. The FAOS judge wasn't the training target — its
   axis didn't get pulled.
2. **Floor / ceiling compression.** FAOS task_adherence is more lenient:
   the base already sits at 0.63 (vs 0.30 on our grader), so there's only
   0.37 of headroom on the FAOS scale before "perfect". Our grader has a
   wider range to move in.
3. **Per-criterion granularity.** FAOS task_adherence averages a small
   number of binary pass/fail criteria. Our 4-part grader gives smooth
   partial credit across 4 dimensions — so a "got the action right but
   missed two of three dollar amounts" answer scores 0.55 (4-part) but
   1.0 (FAOS, "task adhered to").

**Practical takeaway.** Use FAOS for the cold-start prompt rewrite. Use
your own grader to actually measure what you care about. Don't trust the
optimizer's own scoreboard to be sensitive enough to reward downstream
training.